# All Needed Code

This notebook combines all required lab scripts in one place:
- Data preprocessing
- Data validation
- Feature engineering
- Model training
- Model comparison/classification
- Train 
- Reports Generation


## 1) Data Preprocessing (`src/data/preprocess.py`)

In [2]:
"""Clean raw teen mental-health data."""

import argparse
import json
from pathlib import Path

import pandas as pd
import toml
from pydantic import BaseModel


class DataConfig(BaseModel):
    raw_data_path: str
    cleaned_data_path: str
    target_column: str
    numeric_columns: list[str]
    categorical_columns: list[str]


class ReportsConfig(BaseModel):
    cleaning_log_path: str


class AppConfig(BaseModel):
    data: DataConfig
    reports: ReportsConfig


def load_config(filepath: str) -> AppConfig:
    """Load and validate the TOML config with pydantic."""
    raw_config = toml.load(filepath)
    try:
        return AppConfig.model_validate(raw_config)  # pydantic v2
    except AttributeError:
        return AppConfig.parse_obj(raw_config)  # pydantic v1


def load_raw_data(filepath: str) -> pd.DataFrame:
    """Load raw CSV data."""
    return pd.read_csv(filepath)


def clean_data(df: pd.DataFrame, config: DataConfig) -> tuple[pd.DataFrame, dict]:
    """Impute, cap outliers, and deduplicate while preserving target."""
    cleaned = df.copy()
    log: dict[str, int] = {}

    before_rows = len(cleaned)
    cleaned = cleaned.drop_duplicates()
    log["duplicates_removed"] = before_rows - len(cleaned)

    numeric_cols = [c for c in config.numeric_columns if c in cleaned.columns]
    cat_cols = [c for c in config.categorical_columns if c in cleaned.columns]

    for col in numeric_cols:
        missing_before = int(cleaned[col].isna().sum())
        if missing_before > 0:
            cleaned[col] = cleaned[col].fillna(cleaned[col].median())
        q1 = cleaned[col].quantile(0.25)
        q3 = cleaned[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers = int(((cleaned[col] < lower) | (cleaned[col] > upper)).sum())
        cleaned[col] = cleaned[col].clip(lower=lower, upper=upper)
        log[f"{col}_missing_imputed"] = missing_before
        log[f"{col}_outliers_capped"] = outliers

    for col in cat_cols:
        missing_before = int(cleaned[col].isna().sum())
        if missing_before > 0:
            mode_value = cleaned[col].mode(dropna=True)
            replacement = mode_value.iloc[0] if not mode_value.empty else "unknown"
            cleaned[col] = cleaned[col].fillna(replacement)
        log[f"{col}_missing_imputed"] = missing_before

    required_cols = numeric_cols + cat_cols + [config.target_column]
    required_cols = [c for c in required_cols if c in cleaned.columns]
    cleaned = cleaned[required_cols].dropna()
    log["final_row_count"] = len(cleaned)
    return cleaned, log


def save_data(df: pd.DataFrame, filepath: str) -> None:
    """Save processed DataFrame to CSV."""
    Path(filepath).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(filepath, index=False)
    print(f"Saved processed data -> {filepath}  ({len(df)} rows)")


def save_cleaning_log(payload: dict, filepath: str) -> None:
    """Write cleaning summary to JSON."""
    Path(filepath).parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    print(f"Cleaning log saved -> {filepath}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="configs/config.toml")
    args = parser.parse_args()

    print("args", args.config)

    app_config = load_config(args.config)

    df = load_raw_data(app_config.data.raw_data_path)
    cleaned_df, cleaning_log = clean_data(df, app_config.data)
    save_data(cleaned_df, app_config.data.cleaned_data_path)
    save_cleaning_log(cleaning_log, app_config.reports.cleaning_log_path)


usage: colab_kernel_launcher.py [-h] [--config CONFIG]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-740a1e33-095b-408c-8cb3-a186be260796.json


SystemExit: 2

## 2) Data Validation (`src/data/validate.py`)

In [ ]:
"""Run data validation checks and save a JSON report."""

import argparse
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
import toml
from pydantic import BaseModel


class DataConfig(BaseModel):
    target_column: str
    numeric_columns: list[str]
    categorical_columns: list[str]


class AppConfig(BaseModel):
    data: DataConfig


def load_config(filepath: str) -> AppConfig:
    """Load project config."""
    raw_config = toml.load(filepath)
    try:
        return AppConfig.model_validate(raw_config)
    except AttributeError:
        return AppConfig.parse_obj(raw_config)


def build_validation_report(df: pd.DataFrame, config: DataConfig) -> dict:
    """Build validation metrics inspired by lab 4."""
    missing_by_column = {k: int(v) for k, v in df.isna().sum().to_dict().items()}
    report: dict = {
        "generated_at": datetime.utcnow().isoformat(),
        "row_count": int(len(df)),
        "column_count": int(len(df.columns)),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_by_column": missing_by_column,
        "dtypes": {k: str(v) for k, v in df.dtypes.to_dict().items()},
    }

    numeric_summary: dict[str, dict] = {}
    for col in config.numeric_columns:
        if col in df.columns:
            numeric_summary[col] = {
                "min": float(df[col].min()),
                "max": float(df[col].max()),
                "mean": float(df[col].mean()),
                "std": float(df[col].std()),
            }
    report["numeric_summary"] = numeric_summary

    categorical_summary: dict[str, dict] = {}
    for col in config.categorical_columns:
        if col in df.columns:
            categorical_summary[col] = {
                "unique_count": int(df[col].nunique(dropna=True)),
                "top_values": {
                    k: int(v) for k, v in df[col].value_counts().head(5).to_dict().items()
                },
            }
    report["categorical_summary"] = categorical_summary

    target = config.target_column
    if target in df.columns:
        report["target_distribution"] = {
            k: int(v) for k, v in df[target].value_counts().to_dict().items()
        }
    return report


def save_report(report: dict, filepath: str) -> None:
    """Persist validation report as JSON."""
    Path(filepath).parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)
    print(f"Validation report saved -> {filepath}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="configs/config.toml")
    parser.add_argument("--input", required=True, help="Path to CSV file")
    parser.add_argument("--output", required=True, help="Path to JSON report")
    args = parser.parse_args()

    config = load_config(args.config)
    dataframe = pd.read_csv(args.input)
    report_payload = build_validation_report(dataframe, config.data)
    save_report(report_payload, args.output)


usage: colab_kernel_launcher.py [-h] [--config CONFIG] --input INPUT --output
                                OUTPUT
colab_kernel_launcher.py: error: the following arguments are required: --input, --output
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/lib/python3.12/argparse.py", line 1943, in _parse_known_args2
    namespace, args = self._parse_known_args(args, namespace, intermixed)
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 2230, in _parse_known_args
    raise ArgumentError(None, _('the following arguments are required: %s') %
argparse.ArgumentError: the following arguments are required: --input, --output

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_2923/2856687592.py", line 87, in <cell line: 0>
    args, unknown = parser.parse_known_args()
                    ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 1914, in parse_known_args
    retur

TypeError: object of type 'NoneType' has no len()

## 3) Feature Engineering (`src/features/engineer.py`)

In [ ]:
"""Generate model features from cleaned data."""

import argparse
import json
from pathlib import Path

import pandas as pd
import toml
from pydantic import BaseModel
from sklearn.preprocessing import StandardScaler


class DataConfig(BaseModel):
    cleaned_data_path: str
    featured_data_path: str
    target_column: str
    numeric_columns: list[str]
    categorical_columns: list[str]


class ReportsConfig(BaseModel):
    feature_log_path: str


class AppConfig(BaseModel):
    data: DataConfig
    reports: ReportsConfig


def load_config(filepath: str) -> AppConfig:
    """Load and validate config."""
    raw_config = toml.load(filepath)
    try:
        return AppConfig.model_validate(raw_config)
    except AttributeError:
        return AppConfig.parse_obj(raw_config)


def engineer_features(df: pd.DataFrame, config: DataConfig) -> tuple[pd.DataFrame, dict]:
    """Encode categorical columns, scale numerical columns, and add interactions."""
    work_df = df.copy()
    target = work_df[config.target_column]

    numeric_cols = [c for c in config.numeric_columns if c in work_df.columns]
    cat_cols = [c for c in config.categorical_columns if c in work_df.columns]

    scaler = StandardScaler()
    scaled_numeric = pd.DataFrame(
        scaler.fit_transform(work_df[numeric_cols]),
        columns=[f"{c}_scaled" for c in numeric_cols],
        index=work_df.index,
    )

    encoded_categorical = pd.get_dummies(work_df[cat_cols], drop_first=True)
    feature_df = pd.concat([scaled_numeric, encoded_categorical], axis=1)

    if {"stress_level", "anxiety_level"}.issubset(work_df.columns):
        feature_df["stress_anxiety_interaction"] = (
            work_df["stress_level"] * work_df["anxiety_level"]
        )
    if {"screen_time_before_sleep", "sleep_hours"}.issubset(work_df.columns):
        feature_df["screen_sleep_ratio"] = work_df["screen_time_before_sleep"] / (
            work_df["sleep_hours"] + 1e-6
        )
    if {"daily_social_media_hours", "physical_activity"}.issubset(work_df.columns):
        feature_df["social_activity_gap"] = (
            work_df["daily_social_media_hours"] - work_df["physical_activity"]
        )

    feature_df[config.target_column] = target
    log = {
        "input_rows": int(len(df)),
        "output_rows": int(len(feature_df)),
        "numeric_features_scaled": len(numeric_cols),
        "categorical_features_encoded": len(cat_cols),
        "output_feature_count": int(feature_df.shape[1] - 1),
    }
    return feature_df, log


def save_csv(df: pd.DataFrame, filepath: str) -> None:
    """Save feature table."""
    Path(filepath).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(filepath, index=False)
    print(f"Feature dataset saved -> {filepath}")


def save_log(payload: dict, filepath: str) -> None:
    """Save feature engineering log."""
    Path(filepath).parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    print(f"Feature log saved -> {filepath}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="configs/config.toml")
    args = parser.parse_args()

    app_config = load_config(args.config)
    source_df = pd.read_csv(app_config.data.cleaned_data_path)
    featured_df, feature_log = engineer_features(source_df, app_config.data)
    save_csv(featured_df, app_config.data.featured_data_path)
    save_log(feature_log, app_config.reports.feature_log_path)


## 4) Model Training (`src/models/train.py`)

In [ ]:
"""Train a Random Forest model for churn prediction."""

import argparse
import json
import os
import pickle
from pathlib import Path

import pandas as pd
import toml
from dotenv import load_dotenv
from pydantic import BaseModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split


class DataConfig(BaseModel):
    cleaned_data_path: str
    train_feature_columns: list[str]
    target_column: str
    test_size: float
    random_state: int


class ModelConfig(BaseModel):
    n_estimators: int
    max_depth: int
    model_output_path: str


class ReportsConfig(BaseModel):
    metrics_path: str


class AppConfig(BaseModel):
    data: DataConfig
    model: ModelConfig
    reports: ReportsConfig


def load_config(filepath: str) -> AppConfig:
    """Load and validate the TOML config with pydantic."""
    raw_config = toml.load(filepath)
    try:
        return AppConfig.model_validate(raw_config)  # pydantic v2
    except AttributeError:
        return AppConfig.parse_obj(raw_config)  # pydantic v1


def load_data(filepath: str, feature_columns: list[str], target_column: str):
    """Load processed data and split into features and target."""
    df = pd.read_csv(filepath)
    X = df[feature_columns]
    y = df[target_column]
    return X, y


def train_model(X_train, y_train, config) -> RandomForestClassifier:
    """Train a random forest classifier."""
    if isinstance(config, dict):
        model_cfg = config["model"]
        random_state = config["data"]["random_state"]
    else:
        model_cfg = config.model
        random_state = config.data.random_state
    model = RandomForestClassifier(
        n_estimators=model_cfg["n_estimators"] if isinstance(model_cfg, dict) else model_cfg.n_estimators,
        max_depth=model_cfg["max_depth"] if isinstance(model_cfg, dict) else model_cfg.max_depth,
        random_state=random_state,
    )
    model.fit(X_train, y_train)
    return model


def evaluate_model(model, X_test, y_test) -> dict:
    """Return accuracy and f1 metrics."""
    y_pred = model.predict(X_test)
    return {
        "accuracy": round(accuracy_score(y_test, y_pred), 4),
        "f1_score": round(f1_score(y_test, y_pred, zero_division=0), 4),
    }


def save_model(model, filepath: str) -> None:
    """Pickle the trained model."""
    Path(filepath).parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "wb") as f:
        pickle.dump(model, f)
    print(f"Model saved -> {filepath}")


def save_metrics(metrics: dict, filepath: str) -> None:
    """Save metrics dict to JSON."""
    Path(filepath).parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"Metrics saved -> {filepath}")
    print(f"  accuracy : {metrics['accuracy']}")
    print(f"  f1_score : {metrics['f1_score']}")


if __name__ == "__main__":
    load_dotenv(Path(__file__).parent.parent.parent / ".env")

    environment = os.getenv("ENVIRONMENT", "development2")
    model_version = os.getenv("MODEL_VERSION", "v1")

    print(f"Environment : {environment}")
    print(f"Model version: {model_version}")

    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="configs/config.toml")
    args = parser.parse_args()

    config = load_config(args.config)

    X, y = load_data(
        config.data.cleaned_data_path,
        config.data.train_feature_columns,
        config.data.target_column,
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=config.data.test_size,
        random_state=config.data.random_state,
    )

    model = train_model(X_train, y_train, config)
    metrics = evaluate_model(model, X_test, y_test)

    model_dir = config.model.model_output_path.rsplit("/", 1)[0]
    model_path = f"{model_dir}/model_{model_version}.pkl"
    save_model(model, model_path)
    save_metrics(metrics, config.reports.metrics_path)


## 5) Model Comparison (`src/models/classify.py`)

In [ ]:
"""Benchmark multiple classifiers and select the best one."""

import argparse
import json
import pickle
from pathlib import Path

import pandas as pd
import toml
from pydantic import BaseModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC


class DataConfig(BaseModel):
    featured_data_path: str
    target_column: str
    test_size: float
    random_state: int


class ModelConfig(BaseModel):
    model_output_path: str


class ReportsConfig(BaseModel):
    classification_metrics_path: str


class AppConfig(BaseModel):
    data: DataConfig
    model: ModelConfig
    reports: ReportsConfig


def load_config(filepath: str) -> AppConfig:
    """Load and validate project config."""
    raw_config = toml.load(filepath)
    try:
        return AppConfig.model_validate(raw_config)
    except AttributeError:
        return AppConfig.parse_obj(raw_config)


def evaluate_models(df: pd.DataFrame, config: AppConfig) -> tuple[dict, object]:
    """Train and compare several classifiers."""
    X = df.drop(columns=[config.data.target_column])
    y = df[config.data.target_column]
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=config.data.test_size,
        random_state=config.data.random_state,
        stratify=y,
    )

    models = {
        "logistic_regression": LogisticRegression(max_iter=300),
        "knn": KNeighborsClassifier(n_neighbors=7),
        "svm_rbf": SVC(kernel="rbf", probability=False),
        "random_forest": RandomForestClassifier(
            n_estimators=200, random_state=config.data.random_state
        ),
    }

    scores = {}
    best_name = ""
    best_score = -1.0
    best_model = None

    for name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        metrics = {
            "accuracy": round(accuracy_score(y_test, preds), 4),
            "f1_score": round(f1_score(y_test, preds, zero_division=0), 4),
        }
        scores[name] = metrics
        if metrics["f1_score"] > best_score:
            best_score = metrics["f1_score"]
            best_name = name
            best_model = model

    payload = {
        "models": scores,
        "best_model": best_name,
        "best_f1_score": best_score,
        "test_size": config.data.test_size,
    }
    return payload, best_model


def save_json(payload: dict, filepath: str) -> None:
    """Save classification metrics."""
    Path(filepath).parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    print(f"Classification report saved -> {filepath}")


def save_model(model, filepath: str) -> None:
    """Persist the best classifier."""
    Path(filepath).parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "wb") as f:
        pickle.dump(model, f)
    print(f"Best model saved -> {filepath}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="configs/config.toml")
    args = parser.parse_args()

    app_config = load_config(args.config)
    features_df = pd.read_csv(app_config.data.featured_data_path)
    metrics_payload, winner_model = evaluate_models(features_df, app_config)
    save_json(metrics_payload, app_config.reports.classification_metrics_path)
    save_model(winner_model, app_config.model.model_output_path)


## 6) Report Generation (`src/reports/generate_report.py`)

In [ ]:
"""Generate a markdown report for the full ML pipeline."""

import argparse
import json
from datetime import datetime
from pathlib import Path

import toml
from pydantic import BaseModel


class ReportsConfig(BaseModel):
    validation_raw_path: str
    validation_cleaned_path: str
    cleaning_log_path: str
    feature_log_path: str
    metrics_path: str
    classification_metrics_path: str
    pipeline_report_path: str


class AppConfig(BaseModel):
    reports: ReportsConfig


def load_config(filepath: str) -> AppConfig:
    """Load project config."""
    raw_config = toml.load(filepath)
    try:
        return AppConfig.model_validate(raw_config)
    except AttributeError:
        return AppConfig.parse_obj(raw_config)


def load_json(filepath: str) -> dict:
    """Load JSON helper."""
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)


def render_report(
    raw_validation: dict,
    cleaned_validation: dict,
    cleaning_log: dict,
    feature_log: dict,
    baseline_metrics: dict,
    classification_metrics: dict,
) -> str:
    """Create markdown report text."""
    return f"""# End-to-End ML Pipeline Report

Generated at: {datetime.utcnow().isoformat()} UTC

## 1) Validation Before Cleaning
- Rows: {raw_validation.get("row_count")}
- Duplicate rows: {raw_validation.get("duplicate_rows")}
- Missing values total: {sum(raw_validation.get("missing_by_column", {}).values())}

## 2) Cleaning Summary
- Duplicates removed: {cleaning_log.get("duplicates_removed")}
- Final cleaned rows: {cleaning_log.get("final_row_count")}

## 3) Validation After Cleaning
- Rows: {cleaned_validation.get("row_count")}
- Duplicate rows: {cleaned_validation.get("duplicate_rows")}
- Missing values total: {sum(cleaned_validation.get("missing_by_column", {}).values())}

## 4) Feature Engineering
- Input rows: {feature_log.get("input_rows")}
- Output rows: {feature_log.get("output_rows")}
- Output feature count: {feature_log.get("output_feature_count")}

## 5) Baseline Training
- Baseline model accuracy: {baseline_metrics.get("accuracy")}
- Baseline model F1: {baseline_metrics.get("f1_score")}

## 6) Classifier Benchmark
- Best model: {classification_metrics.get("best_model")}
- Best F1 score: {classification_metrics.get("best_f1_score")}

## 7) Notes
- This report is generated automatically from pipeline artifacts in `reports/`.
- You can extend this section with visual EDA charts from notebook lab 7.
"""


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="configs/config.toml")
    args = parser.parse_args()

    app_config = load_config(args.config)
    reports = app_config.reports

    raw_validation = load_json(reports.validation_raw_path)
    cleaned_validation = load_json(reports.validation_cleaned_path)
    cleaning_log = load_json(reports.cleaning_log_path)
    feature_log = load_json(reports.feature_log_path)
    baseline_metrics = load_json(reports.metrics_path)
    classification_metrics = load_json(reports.classification_metrics_path)

    report_text = render_report(
        raw_validation,
        cleaned_validation,
        cleaning_log,
        feature_log,
        baseline_metrics,
        classification_metrics,
    )
    Path(reports.pipeline_report_path).parent.mkdir(parents=True, exist_ok=True)
    Path(reports.pipeline_report_path).write_text(report_text, encoding="utf-8")
    print(f"Pipeline report saved -> {reports.pipeline_report_path}")

## 7) Train (`src/models/classify.py`)

In [ ]:
"""Train a Random Forest model for churn prediction."""

import argparse
import json
import os
import pickle
from pathlib import Path

import pandas as pd
import toml
from dotenv import load_dotenv
from pydantic import BaseModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split


class DataConfig(BaseModel):
    cleaned_data_path: str
    train_feature_columns: list[str]
    target_column: str
    test_size: float
    random_state: int


class ModelConfig(BaseModel):
    n_estimators: int
    max_depth: int
    model_output_path: str


class ReportsConfig(BaseModel):
    metrics_path: str


class AppConfig(BaseModel):
    data: DataConfig
    model: ModelConfig
    reports: ReportsConfig


def load_config(filepath: str) -> AppConfig:
    """Load and validate the TOML config with pydantic."""
    raw_config = toml.load(filepath)
    try:
        return AppConfig.model_validate(raw_config)  # pydantic v2
    except AttributeError:
        return AppConfig.parse_obj(raw_config)  # pydantic v1


def load_data(filepath: str, feature_columns: list[str], target_column: str):
    """Load processed data and split into features and target."""
    df = pd.read_csv(filepath)
    X = df[feature_columns]
    y = df[target_column]
    return X, y


def train_model(X_train, y_train, config) -> RandomForestClassifier:
    """Train a random forest classifier."""
    if isinstance(config, dict):
        model_cfg = config["model"]
        random_state = config["data"]["random_state"]
    else:
        model_cfg = config.model
        random_state = config.data.random_state
    model = RandomForestClassifier(
        n_estimators=(
            model_cfg["n_estimators"]
            if isinstance(model_cfg, dict)
            else model_cfg.n_estimators
        ),
        max_depth=(
            model_cfg["max_depth"] if isinstance(model_cfg, dict) else model_cfg.max_depth
        ),
        random_state=random_state,
    )
    model.fit(X_train, y_train)
    return model


def evaluate_model(model, X_test, y_test) -> dict:
    """Return accuracy and f1 metrics."""
    y_pred = model.predict(X_test)
    return {
        "accuracy": round(accuracy_score(y_test, y_pred), 4),
        "f1_score": round(f1_score(y_test, y_pred, zero_division=0), 4),
    }


def save_model(model, filepath: str) -> None:
    """Pickle the trained model."""
    Path(filepath).parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "wb") as f:
        pickle.dump(model, f)
    print(f"Model saved → {filepath}")


def save_metrics(metrics: dict, filepath: str) -> None:
    """Save metrics dict to JSON."""
    Path(filepath).parent.mkdir(parents=True, exist_ok=True)
    with open(filepath, "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"Metrics saved → {filepath}")
    print(f"  accuracy : {metrics['accuracy']}")
    print(f"  f1_score : {metrics['f1_score']}")


if __name__ == "__main__":
    # Load environment variables from .env
    load_dotenv(Path(__file__).parent.parent.parent / ".env")

    environment = os.getenv("ENVIRONMENT", "development2")
    model_version = os.getenv("MODEL_VERSION", "v1")

    print(f"Environment : {environment}")
    print(f"Model version: {model_version}")

    parser = argparse.ArgumentParser()
    parser.add_argument("--config", default="configs/config.toml")
    args = parser.parse_args()

    config = load_config(args.config)

    X, y = load_data(
        config.data.cleaned_data_path,
        config.data.train_feature_columns,
        config.data.target_column,
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=config.data.test_size,
        random_state=config.data.random_state,
    )

    model = train_model(X_train, y_train, config)
    metrics = evaluate_model(model, X_test, y_test)

    model_dir = config.model.model_output_path.rsplit("/", 1)[0]
    model_path = f"{model_dir}/model_{model_version}.pkl"
    save_model(model, model_path)

    # save_model(model, config["model"]["model_output_path"])
    save_metrics(metrics, config.reports.metrics_path)
